In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from communication import protocol
from communication.rabbitmq import Rabbitmq

# Initialize RabbitMQ connection (adjust parameters as needed)
try:
    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    print("✓ Connected to RabbitMQ successfully")
except Exception as e:
    print(f"✗ Failed to connect to RabbitMQ: {e}")
    print("\nMake sure RabbitMQ is running. You can start it with:")
    print("  python -m startup.start_docker_rabbitmq")

def send_control_message(rmq, msg):
    """Send a control message to the UR3e Mockup via RabbitMQ."""
    try:
        rmq.send_message(
            routing_key=protocol.ROUTING_KEY_CTRL,
            message=msg
        )
        print(f"✓ Control message: {msg} sent successfully")
    except Exception as e:
        print(f"✗ Failed to send control message: {e}")

✓ Connected to RabbitMQ successfully


In [34]:
import numpy as np

# Construct control message for loading a program
def mov_to_pos(position: list, vel: int = 60, acc: int = 80):
    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: position,
        protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
        protocol.CtrlMsgKeys.ACCELERATION: acc,
    }

    send_control_message(rmq, msg)

    # send control message for starting program
    msg_start = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    }

    send_control_message(rmq, msg_start)


In [35]:
position = [0, 0, 0, 0, 0, 0]
mov_to_pos(position)

Message sent to robotarm.ctrl.
✓ Control message: {'type': 'load_program', 'joint_positions': [0, 0, 0, 0, 0, 0], 'max_velocity': 60, 'acceleration': 80} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'play'} sent successfully


In [ ]:
def inject_wear():
    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.INJECT_FAULT,
        protocol.CtrlMsgKeys.FAULT_TYPE: protocol.FaultTypes.WEAR,
        protocol.CtrlMsgKeys.JOINTS: [0, 1, 2, 3, 4, 5],
        protocol.CtrlMsgKeys.FAULT_VALUE: 100,
        protocol.CtrlMsgKeys.DURATION: 100,
    }

inject_wear()

In [37]:
import time
import datetime

def perform_move_wear_perform_move():
    mov_to_pos([0, 0, 0, 0, 0, 0])
    time.sleep(10)
    start1 = time.time()
    mov_to_pos([np.pi/2, 0, 0, 0, 0, 0])
    time.sleep(10)
    mov_to_pos([0, 0, 0, 0, 0, 0])
    time.sleep(10)
    inject_wear()
    time.sleep(10)
    start2 = time.time()
    mov_to_pos([np.pi/2, 0, 0, 0, 0, 0])
    time.sleep(10)
    mov_to_pos([0, 0, 0, 0, 0, 0])

    print("Time between starts = ", start2 - start1)

perform_move_wear_perform_move()

Message sent to robotarm.ctrl.
✓ Control message: {'type': 'load_program', 'joint_positions': [0, 0, 0, 0, 0, 0], 'max_velocity': 60, 'acceleration': 80} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'play'} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'load_program', 'joint_positions': [1.5707963267948966, 0, 0, 0, 0, 0], 'max_velocity': 60, 'acceleration': 80} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'play'} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'load_program', 'joint_positions': [0, 0, 0, 0, 0, 0], 'max_velocity': 60, 'acceleration': 80} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'play'} sent successfully
Message sent to robotarm.ctrl.
✓ Control message: {'type': 'load_program', 'joint_positions': [1.5707963267948966, 0, 0, 0, 0, 0], 'max_velocity': 60, 'acceleration': 80} sent successfully
Message sent to robotar